In [ ]:
# import packages we will need
import pandas as pd
import numpy as np
import numpy.matlib as mb
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import mne
import seaborn as sns
import pickle
import os
import sys
import re
import glob as glob
import argparse

# -------------------------------------------------
# -- project folder directory
dataset_dir = '/data/backed_up/shared/HCC/'
csv_dir = '/mnt/cifs/rdss/rdss_kahwang/LC_HCC/Data_EEG/' # Data_EEG/ # Data_Behavioral # Data_MRI
edf_dir = '/mnt/cifs/rdss/rdss_kahwang/LC_HCC/Data_EEG/EDFs/'

# -- set sns theme
sns.set_theme(context='poster', style='whitegrid', )

In [ ]:
""" Load CSV files and create a master file with all subjects """
# Load ouptut csv files for the two tasks and put each in their own respective master files (all subjects in long format)
# - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - - -
# current pilot subjects list: "10280","10966","11069"
# get list of files for each task
to_exclude=["99999","99998","99930","11072","11054","11055","11059"] # always throw 99999 since its never real data
subjs_to_exclude = ['10981','10982','11046','11086','10982','11093','11088','11094','11087','11097','11089','11099',"11074"] # '11096' #  # ["11072", "11054","11055","11059"] 
# 11072 didn't finish (only did 3 blocks)
# 11054, 11055, 11059 did a version with a contingency error

DM_list = glob.glob(os.path.join(csv_dir,("*_block-*.csv"))) #withModelEstimates.csv")))  dataframe_model-pSpP
print(DM_list)
DM_df_list = []
DM_usable_list = []
for cur_DM in sorted(DM_list):
    # load file
    temp_df = pd.read_csv(cur_DM)
    
    # add subject information
    sid=re.search("[0-9]{5}", cur_DM)
    if sid:
        temp_df["subject"] = sid.group(0)
    
    # add session information
    # sses=re.search("session-[0-2]{3}", cur_DM)
    # if sses:
    #     temp_df["session"] = int(sses.group(0)[-3:])
    
    # add to df list
    if sid.group(0) not in to_exclude:
        DM_df_list.append(temp_df)
        if sid.group(0) not in subjs_to_exclude:
            DM_usable_list.append(temp_df)
DM_df = pd.concat(DM_df_list, ignore_index=True) # merge dfs in list
DM_df_usable = pd.concat(DM_usable_list, ignore_index=True)
# remove no response rows (do not consider trials where no response was made)
#DM_df = DM_df[DM_df['RT']>0]
print("master data frames generated")
print(len(DM_df.subject.unique()), "subjects completed the HCC task")
print(len(DM_df_usable.subject.unique()), "subjects are usable")

DM_df_usable.rename(columns={'subject_correct': 'accuracy', 'hierarhical_switch_type':'hierarchical_switch_type'}, inplace=True)

DM_df_usable

In [ ]:
# create data frame of subject by block performance acc
DM_acc_by_par = DM_df_usable.groupby(['subject','block'])['accuracy'].mean().reset_index()
df_acc = DM_acc_by_par.pivot(index="subject", columns="block", values="accuracy")
#df_acc.to_csv(os.path.join(CSV_dir,"Behavioral_Accuracy_by_Block_and_Subj.csv"))
df_acc

In [ ]:
# add z-scored RT and switch columns
master_list = []
for subject in DM_df_usable['subject'].unique():
    sub_df = DM_df_usable[DM_df_usable['subject']==subject]
    df_list = []
    
    for blk in sub_df['block'].unique():
        blk_df = sub_df[sub_df['block']==blk]
        
        # color & state switch
        cue_shape = np.asarray(blk_df['shape'])
        cue_color = np.asarray(blk_df['color'])
        cue_outline = np.asarray(blk_df['outline'])
        state = np.asarray(blk_df['context'])
        task = np.asarray(blk_df['task'])
        resp = np.asarray(blk_df['subject_response'])
        accuracy = np.asarray(blk_df['accuracy'])
        shape_switch = np.ones(len(cue_color))*np.nan
        color_switch = np.ones(len(cue_color))*np.nan
        outline_switch = np.ones(len(cue_color))*np.nan
        state_switch = np.ones(len(state))*np.nan
        task_switch = np.ones(len(task))*np.nan
        resp_switch = np.ones(len(resp))*np.nan
        prev_accuracy = np.ones(len(accuracy))*np.nan
        prev_state_switch = np.ones(len(state))*np.nan
        state_count = np.zeros(len(state))
        prev_task = np.asarray(np.repeat('XXX',len(state)), dtype=object)
        prev_context = np.asarray(np.repeat('XXX',len(state)), dtype=object)
        
        for ii in range(len(state)):
            if ii>0:
                if cue_shape[int(ii-1)] == cue_shape[ii]:
                    shape_switch[ii] = 0
                else:
                    shape_switch[ii] = 1
                
                if cue_color[int(ii-1)] == cue_color[ii]:
                    color_switch[ii] = 0
                else:
                    color_switch[ii] = 1
                
                if cue_outline[int(ii-1)] == cue_outline[ii]:
                    outline_switch[ii] = 0
                else:
                    outline_switch[ii] = 1
                
                if state[int(ii-1)] == state[ii]:
                    state_switch[ii] = 0
                    state_count[ii] = state_count[int(ii-1)] + 1
                else:
                    state_switch[ii] = 1
                    state_count[ii] = 0
                    
                if task[int(ii-1)] == task[ii]:
                    task_switch[ii] = 0
                else:
                    task_switch[ii] = 1
                    
                if resp[int(ii-1)] == resp[ii]:
                    resp_switch[ii] = 0
                else:
                    resp_switch[ii] = 1
                    
                prev_accuracy[ii] = accuracy[int(ii-1)]
                prev_state_switch[ii] = state_switch[int(ii-1)]
                prev_task[ii] = task[int(ii-1)]
                prev_context[ii] = state[int(ii-1)]
            
        blk_df['shape_switch'] = shape_switch
        blk_df['color_switch'] = color_switch
        blk_df['outline_switch'] = outline_switch
        blk_df['context_switch'] = state_switch
        blk_df['task_switch'] = task_switch
        blk_df['response_switch'] = resp_switch
        blk_df['prev_accuracy'] = prev_accuracy
        blk_df['prev_context_switch'] = prev_state_switch
        blk_df['context_count'] = state_count
        blk_df['prev_task'] = prev_task
        blk_df['prev_context'] = prev_context
        
        # z-scored RT
        subj_resp = np.asarray(blk_df['subject_response'])
        acc = np.asarray(blk_df['accuracy'])
        prev_acc = np.asarray(blk_df['prev_accuracy'])
        rt = np.asarray(blk_df['response_time'])
        zRT = np.ones(len(subj_resp))*np.nan
        for sr in np.unique(subj_resp):
            if np.isnan(sr):
                continue # skip calculation for missed responses, leave as NaN
            sr_inds = np.where(subj_resp==sr)[0]
            acc_inds = np.where(acc==1)[0]
            prev_acc_inds = np.where(prev_acc==1)[0]
            c_mean = np.nanmean( rt[ np.intersect1d(sr_inds, np.intersect1d(acc_inds,prev_acc_inds)) ] )
            c_std = rt[ np.intersect1d(sr_inds, np.intersect1d(acc_inds,prev_acc_inds)) ].std()
            if np.isnan(c_mean):
                print("ERROR mean is NaN")
                print(rt[ np.intersect1d(sr_inds, np.intersect1d(acc_inds,prev_acc_inds)) ])
            if np.isnan(c_std):
                print("ERROR std dev is NaN")
                print(rt[ np.intersect1d(sr_inds, np.intersect1d(acc_inds,prev_acc_inds)) ])
            zRT[sr_inds] = (rt[sr_inds] - c_mean) / c_std
        blk_df['zscored_RT'] = zRT
        
        blk_df['response_time_diff'] = np.concatenate( [[0], rt[1:] - rt[:-1]] )
        blk_df['zscored_RT_diff'] = np.concatenate( [[0], zRT[1:] - zRT[:-1]] )
        
        df_list.append(blk_df)
    
    # save subject to BIDS folder
    subject_df = pd.concat(df_list)
    subject_df = subject_df.sort_values(by=['subject', 'block', 'trial'], ascending=True).reset_index()
    # try:
    #     if not(os.path.exists(os.path.join(BIDS_dir, ("sub-"+subject), 'ses-01', 'beh'))):
    #         os.mkdir(os.path.join(BIDS_dir, ("sub-"+subject), 'ses-01', 'beh'))
    #     subject_df.to_csv(os.path.join(BIDS_dir, ("sub-"+subject), 'ses-01', 'beh', f"sub-{subject}_ses-01_task-HCC2L_beh.tsv"))
    # except:
    #     print("no EEG folder for subject ", subject)
        
    master_list.append(subject_df.iloc[:, 1:])
    
master_df = pd.concat(master_list)
master_df = master_df.sort_values(by=['subject', 'block', 'trial'], ascending=True).reset_index()
master_df

In [ ]:
# set up for plots
plot_df = master_df[master_df['trial']>1]
plot_df = plot_df[((plot_df['zscored_RT']>=-3)&(plot_df['zscored_RT']<=3))]
plot_df = plot_df[plot_df['response_time']>0]

rt_plot_df = plot_df[plot_df['accuracy']==1]
rt_plot_df = rt_plot_df[rt_plot_df['prev_accuracy']==1]

plot_df = plot_df.astype({'block': int})
rt_plot_df = rt_plot_df.astype({'block': int})

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','block'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','block'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','block'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='block', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.6,1.0))
#axes[0].legend_.remove()

# subplot 4 will be RT
sns.boxplot(x='block', y='response_time', data=plot_rt, ax=axes[1])
axes[1].set_title('Response Time')

# subplot 5 will be zRT
sns.boxplot(x='block', y='zscored_RT', data=plot_zrt, ax=axes[2])
axes[2].set_title('zscored_RT')
axes[2].set_ylim((-.25,.25))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 6)) # set up subplots
plot_acc = plot_df.groupby(['subject','task'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','task'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','task'])['zscored_RT'].mean().reset_index() # prep_df
# create rainbow colors for each subject (hard coding for now)
color_list = ['firebrick','coral','gold','greenyellow','olive','cadetblue','turquoise','deepskyblue','royalblue','blue','midnightblue','indigo','darkorchid','orchid','pink','deeppink','mediumvioletred']

# subplot 1 will be accuracy
sns.barplot(x='task', y='accuracy', data=plot_acc, ax=axes[0], order=['T1','T2','T3'])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='task', y='response_time', data=plot_rt, ax=axes[1], order=['T1','T2','T3'])
for sidx, subject_id in enumerate(plot_rt['subject'].unique()):
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='task')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['task'], subject_data['response_time'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['task'], subject_data['response_time'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='task', y='zscored_RT', data=plot_zrt, ax=axes[2], order=['T1','T2','T3'])
for sidx, subject_id in enumerate(plot_zrt['subject'].unique()):
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='task')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['task'], subject_data['zscored_RT'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['task'], subject_data['zscored_RT'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.5,0.5))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=len(master_df.subject.unique()), ncols=3, figsize=(20, 5*len(master_df.subject.unique())), sharex=True) # set up subplots

for sidx, sub in enumerate(master_df.subject.unique()):
    
    plot_acc = plot_df[plot_df['subject']==sub].groupby(['block','task'])['accuracy'].mean().reset_index() # prep_df
    plot_rt = rt_plot_df[rt_plot_df['subject']==sub].groupby(['block','task'])['response_time'].mean().reset_index() # prep_df
    plot_zrt = rt_plot_df[rt_plot_df['subject']==sub].groupby(['block','task'])['zscored_RT'].mean().reset_index() # prep_df

    
    # subplot 1 will be accuracy
    sns.barplot(x='task', y='accuracy', data=plot_acc, ax=axes[sidx][0], order=['T1','T2','T3'])
    axes[sidx][0].set_title('ACCURACY')
    axes[sidx][0].set_ylim((.85,1.0))

    # subplot 2 will be response initiation
    sns.boxplot(x='task', y='response_time', data=plot_rt, ax=axes[sidx][1], order=['T1','T2','T3'])
    for bidx, blk in enumerate(plot_rt['block'].unique()):
        try:
            subject_data = plot_rt[plot_rt['block'] == blk].sort_values(by='task')
            if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
                axes[sidx][1].plot(subject_data['task'], subject_data['response_time'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
            else:
                axes[sidx][1].plot(subject_data['task'], subject_data['response_time'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
        except:
            print("subject had less than 20 blocks")
    axes[sidx][1].set_title(f"{sub} RT")
    axes[sidx][1].set_ylabel("")
    axes[sidx][1].set_ylim((0.400,0.900))

    # subplot 3 will be response initiation
    sns.boxplot(x='task', y='zscored_RT', data=plot_zrt, ax=axes[sidx][2], order=['T1','T2','T3'])
    for bidx, blk in enumerate(plot_zrt['block'].unique()):
        try:
            subject_data = plot_zrt[plot_zrt['block'] == blk].sort_values(by='task')
            if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
                axes[sidx][2].plot(subject_data['task'], subject_data['zscored_RT'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
            else:
                axes[sidx][2].plot(subject_data['task'], subject_data['zscored_RT'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
        except:
            print("less than 20 blocks")
    axes[sidx][2].set_title(f"{sub} zRT")
    axes[sidx][2].set_ylabel("")
    axes[sidx][2].set_ylim((-0.5,0.5))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 6)) # set up subplots
plot_acc = plot_df.groupby(['subject','context'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','context'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','context'])['zscored_RT'].mean().reset_index() # prep_df
# create rainbow colors for each subject (hard coding for now)
color_list = ['firebrick','coral','gold','greenyellow','olive','cadetblue','turquoise','deepskyblue','royalblue','blue','midnightblue','indigo','darkorchid','orchid','pink','deeppink','mediumvioletred']

# subplot 1 will be accuracy
sns.barplot(x='context', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='context', y='response_time', data=plot_rt, ax=axes[1])
for sidx, subject_id in enumerate(plot_rt['subject'].unique()):
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='context')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['context'], subject_data['response_time'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['context'], subject_data['response_time'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='context', y='zscored_RT', data=plot_zrt, ax=axes[2])
for sidx, subject_id in enumerate(plot_zrt['subject'].unique()):
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='context')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['context'], subject_data['zscored_RT'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['context'], subject_data['zscored_RT'], color=color_list[sidx], linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.5,0.5))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=len(master_df.subject.unique()), ncols=3, figsize=(20, 5*len(master_df.subject.unique())), sharex=True) # set up subplots

for sidx, sub in enumerate(master_df.subject.unique()):
    
    plot_acc = plot_df[plot_df['subject']==sub].groupby(['block','context'])['accuracy'].mean().reset_index() # prep_df
    plot_rt = rt_plot_df[rt_plot_df['subject']==sub].groupby(['block','context'])['response_time'].mean().reset_index() # prep_df
    plot_zrt = rt_plot_df[rt_plot_df['subject']==sub].groupby(['block','context'])['zscored_RT'].mean().reset_index() # prep_df

    
    # subplot 1 will be accuracy
    sns.barplot(x='context', y='accuracy', data=plot_acc, ax=axes[sidx][0])
    axes[sidx][0].set_title('ACCURACY')
    axes[sidx][0].set_ylim((.5,1.0))

    # subplot 2 will be response initiation
    sns.boxplot(x='context', y='response_time', data=plot_rt, ax=axes[sidx][1])
    for bidx, blk in enumerate(plot_rt['block'].unique()):
        try:
            subject_data = plot_rt[plot_rt['block'] == blk].sort_values(by='context')
            if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
                axes[sidx][1].plot(subject_data['context'], subject_data['response_time'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
            else:
                axes[sidx][1].plot(subject_data['context'], subject_data['response_time'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
        except:
            print("")
    axes[sidx][1].set_title(f"{sub} RT")
    axes[sidx][1].set_ylabel("")
    axes[sidx][1].set_ylim((0.400,0.900))

    # subplot 3 will be response initiation
    sns.boxplot(x='context', y='zscored_RT', data=plot_zrt, ax=axes[sidx][2])
    for bidx, blk in enumerate(plot_zrt['block'].unique()):
        try:
            subject_data = plot_zrt[plot_zrt['block'] == blk].sort_values(by='context')
            if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
                axes[sidx][2].plot(subject_data['context'], subject_data['zscored_RT'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
            else:
                axes[sidx][2].plot(subject_data['context'], subject_data['zscored_RT'], color=color_list[bidx], linestyle='--', alpha=0.45, zorder=0)
        except:
            print("")
    axes[sidx][2].set_title(f"{sub} zRT")
    axes[sidx][2].set_ylabel("")
    axes[sidx][2].set_ylim((-0.5,0.5))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','context_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt = plot_df.groupby(['subject','context_switch'])['response_time'].mean().reset_index() # prep_df
plot_zrt = plot_df.groupby(['subject','context_switch'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='context_switch', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.85,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='context_switch', y='response_time', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='context_switch')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['context_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['context_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='context_switch', y='zscored_RT', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='context_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['context_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['context_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.75,0.75))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','prev_context_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt = plot_df.groupby(['subject','prev_context_switch'])['response_time'].mean().reset_index() # prep_df
plot_zrt = plot_df.groupby(['subject','prev_context_switch'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='prev_context_switch', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='prev_context_switch', y='response_time', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='prev_context_switch')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['prev_context_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['prev_context_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='prev_context_switch', y='zscored_RT', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='prev_context_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['prev_context_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['prev_context_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.75,0.75))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','shape_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','shape_switch'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','shape_switch'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='shape_switch', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='shape_switch', y='response_time', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='shape_switch')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['shape_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['shape_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='shape_switch', y='zscored_RT', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='shape_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['shape_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['shape_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.75,0.75))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','color_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','color_switch'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','color_switch'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='color_switch', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='color_switch', y='response_time', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='color_switch')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['color_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['color_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='color_switch', y='zscored_RT', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='color_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['color_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['color_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.75,0.75))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','outline_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','outline_switch'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','outline_switch'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='outline_switch', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='outline_switch', y='response_time', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='outline_switch')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['outline_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['outline_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='outline_switch', y='zscored_RT', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='outline_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['outline_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['outline_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.75,0.75))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','task_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','task_switch'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','task_switch'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='task_switch', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='task_switch', y='response_time', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='task_switch')
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['task_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['task_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((0.400,0.900))

# subplot 3 will be response initiation
sns.boxplot(x='task_switch', y='zscored_RT', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='task_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['task_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['task_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.25,0.25))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 5)) # set up subplots
plot_acc = plot_df.groupby(['subject','task_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','task_switch'])['response_time_diff'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','task_switch'])['zscored_RT_diff'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='task_switch', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.9,1.0))

# subplot 2 will be response initiation
sns.boxplot(x='task_switch', y='response_time_diff', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by='task_switch')
    if list(subject_data['response_time_diff'])[0] - list(subject_data['response_time_diff'])[1] < 0:
        axes[1].plot(subject_data['task_switch'], subject_data['response_time_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['task_switch'], subject_data['response_time_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT')
axes[1].set_ylim((-0.5,0.5))

# subplot 3 will be response initiation
sns.boxplot(x='task_switch', y='zscored_RT_diff', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='task_switch')
    if list(subject_data['zscored_RT_diff'])[0] - list(subject_data['zscored_RT_diff'])[1] < 0:
        axes[2].plot(subject_data['task_switch'], subject_data['zscored_RT_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['task_switch'], subject_data['zscored_RT_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT')
axes[2].set_ylim((-0.5,0.5))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=3, ncols=1, figsize=(15, 15)) # set up subplots
plot_acc_tmp = plot_df.groupby(['subject','block','task_switch'])['accuracy'].mean().reset_index() # prep_df
plot_rt_tmp = rt_plot_df.groupby(['subject','block','task_switch'])['response_time'].mean().reset_index() # prep_df
plot_zrt_tmp = rt_plot_df.groupby(['subject','block','task_switch'])['zscored_RT'].mean().reset_index() # prep_df
plot_acc = plot_df.groupby(['subject','block'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','block'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','block'])['zscored_RT'].mean().reset_index() # prep_df
plot_acc['accuracy'] = np.asarray(plot_acc_tmp["accuracy"][1:192:2]) - np.asarray(plot_acc_tmp["accuracy"][0:192:2]) # left side is task switch, right side is task repeat (pos value means task switch cost)
plot_rt['response_time'] = np.asarray(plot_rt_tmp["response_time"][1:192:2]) - np.asarray(plot_rt_tmp["response_time"][0:192:2]) # left side is task switch, right side is task repeat (pos value means task switch cost)
plot_zrt['zscored_RT'] = np.asarray(plot_zrt_tmp["zscored_RT"][1:192:2]) - np.asarray(plot_zrt_tmp["zscored_RT"][0:192:2]) # left side is task switch, right side is task repeat (pos value means task switch cost)
plot_acc['block'] = plot_acc['block'].astype(int)
plot_rt['block'] = plot_rt['block'].astype(int)
plot_zrt['block'] = plot_zrt['block'].astype(int)

# subplot 1 will be accuracy
sns.barplot(x='block', y='accuracy', data=plot_acc, ax=axes[0])
axes[0].set_title('ACCURACY (task switch - task repeat)')
axes[0].set_ylim((-.1,0.1))

# subplot 2 will be response initiation
sns.boxplot(x='block', y='response_time', data=plot_rt, ax=axes[1])
for subject_id in plot_rt['subject'].unique():
    subject_data = plot_rt[plot_rt['subject'] == subject_id].sort_values(by=['block']).reset_index(drop=True)
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['block'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['block'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('RT (task switch - task repeat)')
axes[1].set_ylim((-.3,0.3))

# subplot 3 will be response initiation
sns.boxplot(x='block', y='zscored_RT', data=plot_zrt, ax=axes[2])
for subject_id in plot_zrt['subject'].unique():
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by=['block']).reset_index(drop=True)
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['block'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['block'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zRT (task switch - task repeat)')
axes[2].set_ylim((-0.95,0.95))


# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
ids_palette = ['#56B4E9', '#56B4E9']
# group by subject
ids_df = rt_plot_df[rt_plot_df['hierarchical_switch_type']=='IDS']
ids_zRT_df = ids_df.groupby(['subject','shape_switch'])['zscored_RT'].mean().reset_index() # get avg value of category per subject
#ids_zRT_df['shape_switch'] = np.where(ids_zRT_df['shape_switch']==1, '1 feature change', '2 feature change')

eds_palette = ['#E69F00', '#E69F00']
# group by subject
eds_df = rt_plot_df[rt_plot_df['hierarchical_switch_type']=='EDS']
eds_zRT_df = eds_df.groupby(['subject','shape_switch'])['zscored_RT'].mean().reset_index() # get avg value of category per subject
#eds_zRT_df['shape_switch'] = np.where(eds_zRT_df['shape_switch']==1, '1 feature change', '2 feature change')

# set up figure for subplots
fig, axes = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(8, 5)) # set up subplots


# subplot 2 will be IDS
sns.boxplot(x='shape_switch', y='zscored_RT', data=ids_zRT_df, ax=axes[0], palette=ids_palette, boxprops=dict(alpha=.75))
sns.stripplot(x='shape_switch', y='zscored_RT', data=ids_zRT_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[0], order=['1 feature change','2 feature change'])
# Manually draw lines between subjects
for subject_id in ids_zRT_df['subject'].unique():
    subject_data = ids_zRT_df[ids_zRT_df['subject'] == subject_id].sort_values(by='shape_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[0].plot(subject_data['shape_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['shape_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_title('zRT')
#axes[0].tick_params(axis='x', rotation=45)
EDS_avg = round(ids_zRT_df.loc[(ids_zRT_df['shape_switch']==1), "zscored_RT"].mean(), 3)
IDS_avg = round(ids_zRT_df.loc[(ids_zRT_df['shape_switch']==0), "zscored_RT"].mean(), 3)
custom_labels = [f"shape repeat\nM={IDS_avg}", f"shape switch\nM={EDS_avg}"]
axes[0].set_xticklabels(custom_labels,fontsize=7.5)
axes[0].set_ylim((-.4,0.95))

# subplot 2 will be EDS
sns.boxplot(x='shape_switch', y='zscored_RT', data=eds_zRT_df, ax=axes[1], palette=eds_palette, boxprops=dict(alpha=.75))
sns.stripplot(x='shape_switch', y='zscored_RT', data=eds_zRT_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[1], order=['1 feature change','2 feature change'])
# Manually draw lines between subjects
for subject_id in eds_zRT_df['subject'].unique():
    subject_data = eds_zRT_df[eds_zRT_df['subject'] == subject_id].sort_values(by='shape_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[1].plot(subject_data['shape_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['shape_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('zRT')
#axes[1].tick_params(axis='x', rotation=45)
EDS_avg = round(eds_zRT_df.loc[(eds_zRT_df['shape_switch']==1), "zscored_RT"].mean(), 3)
IDS_avg = round(eds_zRT_df.loc[(eds_zRT_df['shape_switch']==0), "zscored_RT"].mean(), 3)
custom_labels = [f"shape repeat\nM={IDS_avg}", f"shape switch\nM={EDS_avg}"]
axes[1].set_xticklabels(custom_labels,fontsize=7.5)
axes[1].set_ylim((-.4,0.95))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
ids_palette = ['#56B4E9', '#56B4E9']
# group by subject
ids_df = rt_plot_df[rt_plot_df['hierarchical_switch_type']=='IDS']
ids_zRT_df = ids_df.groupby(['subject','color_switch'])['zscored_RT'].mean().reset_index() # get avg value of category per subject
#ids_zRT_df['color_switch'] = np.where(ids_zRT_df['color_switch']==1, '1 feature change', '2 feature change')

eds_palette = ['#E69F00', '#E69F00']
# group by subject
eds_df = rt_plot_df[rt_plot_df['hierarchical_switch_type']=='EDS']
eds_zRT_df = eds_df.groupby(['subject','color_switch'])['zscored_RT'].mean().reset_index() # get avg value of category per subject
#eds_zRT_df['color_switch'] = np.where(eds_zRT_df['color_switch']==1, '1 feature change', '2 feature change')

# set up figure for subplots
fig, axes = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(8, 5)) # set up subplots


# subplot 2 will be IDS
sns.boxplot(x='color_switch', y='zscored_RT', data=ids_zRT_df, ax=axes[0], palette=ids_palette, boxprops=dict(alpha=.75))
sns.stripplot(x='color_switch', y='zscored_RT', data=ids_zRT_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[0], order=['1 feature change','2 feature change'])
# Manually draw lines between subjects
for subject_id in ids_zRT_df['subject'].unique():
    subject_data = ids_zRT_df[ids_zRT_df['subject'] == subject_id].sort_values(by='color_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[0].plot(subject_data['color_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['color_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_title('zRT')
#axes[0].tick_params(axis='x', rotation=45)
EDS_avg = round(ids_zRT_df.loc[(ids_zRT_df['color_switch']==1), "zscored_RT"].mean(), 3)
IDS_avg = round(ids_zRT_df.loc[(ids_zRT_df['color_switch']==0), "zscored_RT"].mean(), 3)
custom_labels = [f"color repeat\nM={IDS_avg}", f"color switch\nM={EDS_avg}"]
axes[0].set_xticklabels(custom_labels,fontsize=7.5)
axes[0].set_ylim((-.4,0.95))

# subplot 2 will be EDS
sns.boxplot(x='color_switch', y='zscored_RT', data=eds_zRT_df, ax=axes[1], palette=eds_palette, boxprops=dict(alpha=.75))
sns.stripplot(x='color_switch', y='zscored_RT', data=eds_zRT_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[1], order=['1 feature change','2 feature change'])
# Manually draw lines between subjects
for subject_id in eds_zRT_df['subject'].unique():
    subject_data = eds_zRT_df[eds_zRT_df['subject'] == subject_id].sort_values(by='color_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[1].plot(subject_data['color_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['color_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('zRT')
#axes[1].tick_params(axis='x', rotation=45)
EDS_avg = round(eds_zRT_df.loc[(eds_zRT_df['color_switch']==1), "zscored_RT"].mean(), 3)
IDS_avg = round(eds_zRT_df.loc[(eds_zRT_df['color_switch']==0), "zscored_RT"].mean(), 3)
custom_labels = [f"color repeat\nM={IDS_avg}", f"color switch\nM={EDS_avg}"]
axes[1].set_xticklabels(custom_labels,fontsize=7.5)
axes[1].set_ylim((-.4,0.95))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
ids_palette = ['#56B4E9', '#56B4E9']
# group by subject
ids_df = rt_plot_df[rt_plot_df['hierarchical_switch_type']=='IDS']
ids_zRT_df = ids_df.groupby(['subject','outline_switch'])['zscored_RT'].mean().reset_index() # get avg value of category per subject
#ids_zRT_df['outline_switch'] = np.where(ids_zRT_df['outline_switch']==1, '1 feature change', '2 feature change')

eds_palette = ['#E69F00', '#E69F00']
# group by subject
eds_df = rt_plot_df[rt_plot_df['hierarchical_switch_type']=='EDS']
eds_zRT_df = eds_df.groupby(['subject','outline_switch'])['zscored_RT'].mean().reset_index() # get avg value of category per subject
#eds_zRT_df['outline_switch'] = np.where(eds_zRT_df['outline_switch']==1, '1 feature change', '2 feature change')

# set up figure for subplots
fig, axes = plt.subplots(nrows=1, ncols=2, sharey=True, figsize=(8, 5)) # set up subplots


# subplot 2 will be IDS
sns.boxplot(x='outline_switch', y='zscored_RT', data=ids_zRT_df, ax=axes[0], palette=ids_palette, boxprops=dict(alpha=.75))
sns.stripplot(x='outline_switch', y='zscored_RT', data=ids_zRT_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[0], order=['1 feature change','2 feature change'])
# Manually draw lines between subjects
for subject_id in ids_zRT_df['subject'].unique():
    subject_data = ids_zRT_df[ids_zRT_df['subject'] == subject_id].sort_values(by='outline_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[0].plot(subject_data['outline_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['outline_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_title('zRT')
#axes[0].tick_params(axis='x', rotation=45)
EDS_avg = round(ids_zRT_df.loc[(ids_zRT_df['outline_switch']==1), "zscored_RT"].mean(), 3)
IDS_avg = round(ids_zRT_df.loc[(ids_zRT_df['outline_switch']==0), "zscored_RT"].mean(), 3)
custom_labels = [f"outline repeat\nM={IDS_avg}", f"outline switch\nM={EDS_avg}"]
axes[0].set_xticklabels(custom_labels,fontsize=7.5)
axes[0].set_ylim((-.4,0.95))

# subplot 2 will be EDS
sns.boxplot(x='outline_switch', y='zscored_RT', data=eds_zRT_df, ax=axes[1], palette=eds_palette, boxprops=dict(alpha=.75))
sns.stripplot(x='outline_switch', y='zscored_RT', data=eds_zRT_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[1], order=['1 feature change','2 feature change'])
# Manually draw lines between subjects
for subject_id in eds_zRT_df['subject'].unique():
    subject_data = eds_zRT_df[eds_zRT_df['subject'] == subject_id].sort_values(by='outline_switch')
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[1].plot(subject_data['outline_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['outline_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('zRT')
#axes[1].tick_params(axis='x', rotation=45)
EDS_avg = round(eds_zRT_df.loc[(eds_zRT_df['outline_switch']==1), "zscored_RT"].mean(), 3)
IDS_avg = round(eds_zRT_df.loc[(eds_zRT_df['outline_switch']==0), "zscored_RT"].mean(), 3)
custom_labels = [f"outline repeat\nM={IDS_avg}", f"outline switch\nM={EDS_avg}"]
axes[1].set_xticklabels(custom_labels,fontsize=7.5)
axes[1].set_ylim((-.4,0.95))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
plot_zrt_count = rt_plot_df.groupby(['subject','hierarchical_switch_type','block','cue'])['zscored_RT'].count().reset_index()
plot_zrt_count

fig, axes = plt.subplots(nrows=3, ncols=1, sharex=True, figsize=(25, 12)) # set up subplots

sns.boxplot(x='block', y='zscored_RT', hue='cue', data=plot_zrt_count[plot_zrt_count['hierarchical_switch_type']=='Stay'], ax=axes[0], legend=False)
axes[0].set_ylim((0,9))
sns.boxplot(x='block', y='zscored_RT', hue='cue', data=plot_zrt_count[plot_zrt_count['hierarchical_switch_type']=='IDS'], ax=axes[1], legend=False)
axes[1].set_ylim((0,9))
sns.boxplot(x='block', y='zscored_RT', hue='cue', data=plot_zrt_count[plot_zrt_count['hierarchical_switch_type']=='EDS'], ax=axes[2], legend=False)
axes[2].set_ylim((0,9))

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
plot_zrt_count

In [ ]:
plot_acc = plot_df.groupby(['subject','hierarchical_switch_type','block','cue'])['accuracy'].mean().reset_index()
plot_acc_count = plot_df.groupby(['subject','hierarchical_switch_type','block','cue'])['accuracy'].count().reset_index()
plot_acc['count'] = plot_acc_count['accuracy']
plot_acc['accuracy'] = plot_acc['accuracy']*9.9 #+ 4

fig, axes = plt.subplots(nrows=6, ncols=1, sharex=True, figsize=(25, 20)) # set up subplots
plot_acc = plot_acc[plot_acc['subject'] == '11098'] # 11098  # 11096

sns.boxplot(x='block', y='count', hue='hierarchical_switch_type', data=plot_acc[plot_acc['cue']=='bgm'], ax=axes[0], legend=False)
for subject_id in plot_acc['hierarchical_switch_type'].unique():
    subject_data = plot_acc[((plot_acc['cue']=='bgm')&(plot_acc['hierarchical_switch_type'] == subject_id))].sort_values(by='block')
    if list(subject_data['accuracy'])[0] - list(subject_data['accuracy'])[1] < 0:
        axes[0].plot(subject_data['block'], subject_data['accuracy'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['block'], subject_data['accuracy'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_ylim((0,10))

sns.boxplot(x='block', y='count', hue='hierarchical_switch_type', data=plot_acc[plot_acc['cue']=='bvs'], ax=axes[1], legend=False)
for subject_id in plot_acc['hierarchical_switch_type'].unique():
    subject_data = plot_acc[((plot_acc['cue']=='bvs')&(plot_acc['hierarchical_switch_type'] == subject_id))].sort_values(by='block')
    if list(subject_data['accuracy'])[0] - list(subject_data['accuracy'])[1] < 0:
        axes[1].plot(subject_data['block'], subject_data['accuracy'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['block'], subject_data['accuracy'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_ylim((0,10))

sns.boxplot(x='block', y='count', hue='hierarchical_switch_type', data=plot_acc[plot_acc['cue']=='grm'], ax=axes[2], legend=False)
for subject_id in plot_acc['hierarchical_switch_type'].unique():
    subject_data = plot_acc[((plot_acc['cue']=='grm')&(plot_acc['hierarchical_switch_type'] == subject_id))].sort_values(by='block')
    if list(subject_data['accuracy'])[0] - list(subject_data['accuracy'])[1] < 0:
        axes[2].plot(subject_data['block'], subject_data['accuracy'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['block'], subject_data['accuracy'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_ylim((0,10))

sns.boxplot(x='block', y='count', hue='hierarchical_switch_type', data=plot_acc[plot_acc['cue']=='gvl'], ax=axes[3], legend=False)
for subject_id in plot_acc['hierarchical_switch_type'].unique():
    subject_data = plot_acc[((plot_acc['cue']=='gvl')&(plot_acc['hierarchical_switch_type'] == subject_id))].sort_values(by='block')
    if list(subject_data['accuracy'])[0] - list(subject_data['accuracy'])[1] < 0:
        axes[3].plot(subject_data['block'], subject_data['accuracy'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[3].plot(subject_data['block'], subject_data['accuracy'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[3].set_ylim((0,10))

sns.boxplot(x='block', y='count', hue='hierarchical_switch_type', data=plot_acc[plot_acc['cue']=='hgl'], ax=axes[4], legend=False)
for subject_id in plot_acc['hierarchical_switch_type'].unique():
    subject_data = plot_acc[((plot_acc['cue']=='hgl')&(plot_acc['hierarchical_switch_type'] == subject_id))].sort_values(by='block')
    if subject_id == 'Stay':
        axes[4].plot(subject_data['block'], subject_data['accuracy'], color='green', linestyle='--', alpha=0.45, zorder=0)
    elif subject_id == 'IDS':
        axes[4].plot(subject_data['block'], subject_data['accuracy'], color='cyan', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[4].plot(subject_data['block'], subject_data['accuracy'], color='gold', linestyle='--', alpha=0.45, zorder=0)
axes[4].set_ylim((0,10))

sns.boxplot(x='block', y='count', hue='hierarchical_switch_type', data=plot_acc[plot_acc['cue']=='hrs'], ax=axes[5], legend=False)
for subject_id in plot_acc['hierarchical_switch_type'].unique():
    subject_data = plot_acc[((plot_acc['cue']=='hrs')&(plot_acc['hierarchical_switch_type'] == subject_id))].sort_values(by='block')
    if list(subject_data['accuracy'])[0] - list(subject_data['accuracy'])[1] < 0:
        axes[5].plot(subject_data['block'], subject_data['accuracy'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[5].plot(subject_data['block'], subject_data['accuracy'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[5].set_ylim((0,10))


# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
plot_CrCs_count = rt_plot_df.groupby(['subject','context_switch','task_switch','response_switch'])['response_time'].count().reset_index() # get avg value of category per subject
plot_CrCs_count


In [ ]:
custom_palette = ['#009E73', '#56B4E9', '#E69F00'] # task switch palette = ['#888888', '#D55E00']
sns.set_palette(custom_palette)
plot_df['hierarchical_switch_type'] = pd.Categorical(plot_df['hierarchical_switch_type'], categories=['Stay','IDS','EDS'], ordered=True)
rt_plot_df['hierarchical_switch_type'] = pd.Categorical(rt_plot_df['hierarchical_switch_type'], categories=['Stay','IDS','EDS'], ordered=True)

# PLOTS PLOTS PLOTS
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(20, 6)) # set up subplots
plot_acc = plot_df.groupby(['subject','hierarchical_switch_type'])['accuracy'].mean().reset_index() # prep_df
plot_rt = rt_plot_df.groupby(['subject','hierarchical_switch_type'])['response_time'].mean().reset_index() # prep_df
plot_zrt = rt_plot_df.groupby(['subject','hierarchical_switch_type'])['zscored_RT'].mean().reset_index() # prep_df

# subplot 1 will be accuracy
sns.barplot(x='hierarchical_switch_type', y='accuracy', hue='hierarchical_switch_type', data=plot_acc, ax=axes[0], order=['Stay','IDS','EDS'])
axes[0].set_title('ACCURACY')
axes[0].set_ylim((.85,1.0))
axes[0].set_xticklabels(["Stay","IDS","EDS"],fontsize=15)

# subplot 2 will be response initiation
sns.boxplot(x='hierarchical_switch_type', y='response_time', hue='hierarchical_switch_type', data=plot_rt, showmeans=True, ax=axes[1], order=['Stay','IDS','EDS'])
axes[1].set_title('RT',fontsize=15)
EDS_avg = plot_rt.loc[((plot_rt['hierarchical_switch_type']=="EDS")), "response_time"].mean().round(3)
IDS_avg = plot_rt.loc[((plot_rt['hierarchical_switch_type']=="IDS")), "response_time"].mean().round(3)
repeat_avg = plot_rt.loc[((plot_rt['hierarchical_switch_type']=="Stay")), "response_time"].mean().round(3)
custom_labels = [f"repeat\nM={repeat_avg}", f"IDS switch\nM={IDS_avg}", f"EDS switch\nM={EDS_avg}"]
axes[1].set_xticklabels(custom_labels,fontsize=15)

# subplot 3 will be response initiation
sns.boxplot(x='hierarchical_switch_type', y='zscored_RT', hue='hierarchical_switch_type', data=plot_zrt, showmeans=True, ax=axes[2], order=['Stay','IDS','EDS'])
# Manually draw lines between subjects
for subject_id in plot_zrt['subject'].unique():
    if subject_id == "11074":
        continue
    # if subject_id == "11020":
    #     continue
    subject_data = plot_zrt[plot_zrt['subject'] == subject_id].sort_values(by='hierarchical_switch_type')
    if (list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0)&(list(subject_data['zscored_RT'])[1] - list(subject_data['zscored_RT'])[2] < 0):
        axes[2].plot(subject_data['hierarchical_switch_type'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['hierarchical_switch_type'], subject_data['zscored_RT'], color='lightgrey', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('zscored RT',fontsize=15)
EDS_avg = plot_zrt.loc[((plot_zrt['hierarchical_switch_type']=="EDS")), "zscored_RT"].mean().round(3)
IDS_avg = plot_zrt.loc[((plot_zrt['hierarchical_switch_type']=="IDS")), "zscored_RT"].mean().round(3)
repeat_avg = plot_zrt.loc[((plot_zrt['hierarchical_switch_type']=="Stay")), "zscored_RT"].mean().round(3)
custom_labels = [f"repeat\nM={repeat_avg}", f"IDS switch\nM={IDS_avg}", f"EDS switch\nM={EDS_avg}"]
axes[2].set_xticklabels(custom_labels,fontsize=15)

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
custom_palette = ['#555555', '#D55E00']
sns.set_palette(custom_palette)
# group by subject
plot_CrCs_df = rt_plot_df.groupby(['subject','context_switch','task_switch','response_switch'])['response_time'].mean().reset_index() # get avg value of category per subject

plot_Cr_df = plot_CrCs_df[plot_CrCs_df['context_switch']==0]
plot_Cs_df = plot_CrCs_df[plot_CrCs_df['context_switch']==1]

plot_CrTr_df = plot_Cr_df[plot_Cr_df['task_switch']==0]
plot_CrTs_df = plot_Cr_df[plot_Cr_df['task_switch']==1]
plot_CsTr_df = plot_Cs_df[plot_Cs_df['task_switch']==0]
plot_CsTs_df = plot_Cs_df[plot_Cs_df['task_switch']==1]
# set up figure for subplots
fig, axes = plt.subplots(nrows=1, ncols=4, sharey=True, figsize=(16, 7.5)) # set up subplots


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='response_time', hue='response_switch', data=plot_CrTr_df, ax=axes[0]) # boxprops=dict(alpha=.75), , order=[0,1]
sns.stripplot(x='response_switch', y='response_time', data=plot_CrTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[0], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CrTr_df[plot_CrTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[0].plot(subject_data['response_switch'], subject_data['response_time'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[0].plot(subject_data['response_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['response_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_title('Context Repeat\nTask Repeat')
axes[0].set_ylim((0.425,0.975))
axes[0].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='response_time', hue='response_switch', data=plot_CrTs_df, ax=axes[1]) # boxprops=dict(alpha=.75),  
sns.stripplot(x='response_switch', y='response_time', data=plot_CrTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[1], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CrTs_df[plot_CrTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[1].plot(subject_data['response_switch'], subject_data['response_time'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[1].plot(subject_data['response_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['response_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('Context Repeat\nTask Switch')
axes[1].set_ylim((0.425,0.975))
axes[1].get_legend().remove()


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='response_time', hue='response_switch', data=plot_CsTr_df, ax=axes[2]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='response_time', data=plot_CsTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[2], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CsTr_df[plot_CsTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[2].plot(subject_data['response_switch'], subject_data['response_time'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[2].plot(subject_data['response_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['response_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('Context Switch\nTask Repeat')
axes[2].set_ylim((0.425,0.975))
axes[2].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='response_time', hue='response_switch', data=plot_CsTs_df, ax=axes[3]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='response_time', data=plot_CsTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[3], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CsTs_df[plot_CsTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[3].plot(subject_data['response_switch'], subject_data['response_time'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time'])[0] - list(subject_data['response_time'])[1] < 0:
        axes[3].plot(subject_data['response_switch'], subject_data['response_time'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[3].plot(subject_data['response_switch'], subject_data['response_time'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[3].set_title('Context Switch\nTask Switch')
axes[3].set_ylim((0.425,0.975))
axes[3].get_legend().remove()


# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
custom_palette = ['#555555', '#D55E00']
sns.set_palette(custom_palette)
# group by subject
tmp_df = rt_plot_df[((rt_plot_df['task']!='T3') & (rt_plot_df['prev_task']!='T3'))]

plot_CrCs_df = tmp_df.groupby(['subject','context_switch','task_switch','response_switch'])['zscored_RT'].mean().reset_index() # get avg value of category per subject

plot_Cr_df = plot_CrCs_df[plot_CrCs_df['context_switch']==0]
plot_Cs_df = plot_CrCs_df[plot_CrCs_df['context_switch']==1]

plot_CrTr_df = plot_Cr_df[plot_Cr_df['task_switch']==0]
plot_CrTs_df = plot_Cr_df[plot_Cr_df['task_switch']==1]
plot_CsTr_df = plot_Cs_df[plot_Cs_df['task_switch']==0]
plot_CsTs_df = plot_Cs_df[plot_Cs_df['task_switch']==1]
# set up figure for subplots
fig, axes = plt.subplots(nrows=1, ncols=4, sharey=True, figsize=(16, 7.5)) # set up subplots


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='zscored_RT', hue='response_switch', data=plot_CrTr_df, ax=axes[0]) # boxprops=dict(alpha=.75), , order=[0,1]
sns.stripplot(x='response_switch', y='zscored_RT', data=plot_CrTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[0], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    if subject_id == "11020":
        continue
    if subject_id == "11102":
        continue
    subject_data = plot_CrTr_df[plot_CrTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[0].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[0].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_title('Context Repeat\nTask Repeat')
axes[0].set_ylim((-0.75,0.75))
axes[0].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='zscored_RT', hue='response_switch', data=plot_CrTs_df, ax=axes[1]) # boxprops=dict(alpha=.75),  
sns.stripplot(x='response_switch', y='zscored_RT', data=plot_CrTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[1], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    if subject_id == "11020":
        continue
    if subject_id == "11100":
        continue
    subject_data = plot_CrTs_df[plot_CrTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[1].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[1].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('Context Repeat\nTask Switch')
axes[1].set_ylim((-0.75,0.75))
axes[1].get_legend().remove()


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='zscored_RT', hue='response_switch', data=plot_CsTr_df, ax=axes[2]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='zscored_RT', data=plot_CsTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[2], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    if subject_id == "11020":
        continue
    if subject_id == "11100":
        continue
    subject_data = plot_CsTr_df[plot_CsTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[2].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[2].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('Context Switch\nTask Repeat')
axes[2].set_ylim((-0.75,0.75))
axes[2].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='zscored_RT', hue='response_switch', data=plot_CsTs_df, ax=axes[3]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='zscored_RT', data=plot_CsTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[3], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    if subject_id == "11020":
        continue
    if subject_id == "11100":
        continue
    subject_data = plot_CsTs_df[plot_CsTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[3].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT'])[0] - list(subject_data['zscored_RT'])[1] < 0:
        axes[3].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[3].plot(subject_data['response_switch'], subject_data['zscored_RT'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[3].set_title('Context Switch\nTask Switch')
axes[3].set_ylim((-0.75,0.75))
axes[3].get_legend().remove()


# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
custom_palette = ['#555555', '#D55E00']
sns.set_palette(custom_palette)
# group by subject
plot_CrCs_df = rt_plot_df.groupby(['subject','context_switch','task_switch','response_switch'])['response_time_diff'].mean().reset_index() # get avg value of category per subject

plot_Cr_df = plot_CrCs_df[plot_CrCs_df['context_switch']==0]
plot_Cs_df = plot_CrCs_df[plot_CrCs_df['context_switch']==1]

plot_CrTr_df = plot_Cr_df[plot_Cr_df['task_switch']==0]
plot_CrTs_df = plot_Cr_df[plot_Cr_df['task_switch']==1]
plot_CsTr_df = plot_Cs_df[plot_Cs_df['task_switch']==0]
plot_CsTs_df = plot_Cs_df[plot_Cs_df['task_switch']==1]
# set up figure for subplots
fig, axes = plt.subplots(nrows=1, ncols=4, sharey=True, figsize=(16, 7.5)) # set up subplots


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='response_time_diff', hue='response_switch', data=plot_CrTr_df, ax=axes[0]) # boxprops=dict(alpha=.75), , order=[0,1]
sns.stripplot(x='response_switch', y='response_time_diff', data=plot_CrTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[0], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CrTr_df[plot_CrTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[0].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time_diff'])[0] - list(subject_data['response_time_diff'])[1] < 0:
        axes[0].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_title('Context Repeat\nTask Repeat')
axes[0].set_ylim((-0.25,0.25))
axes[0].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='response_time_diff', hue='response_switch', data=plot_CrTs_df, ax=axes[1]) # boxprops=dict(alpha=.75),  
sns.stripplot(x='response_switch', y='response_time_diff', data=plot_CrTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[1], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CrTs_df[plot_CrTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[1].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time_diff'])[0] - list(subject_data['response_time_diff'])[1] < 0:
        axes[1].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('Context Repeat\nTask Switch')
axes[1].set_ylim((-0.25,0.25))
axes[1].get_legend().remove()


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='response_time_diff', hue='response_switch', data=plot_CsTr_df, ax=axes[2]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='response_time_diff', data=plot_CsTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[2], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CsTr_df[plot_CsTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[2].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time_diff'])[0] - list(subject_data['response_time_diff'])[1] < 0:
        axes[2].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('Context Switch\nTask Repeat')
axes[2].set_ylim((-0.25,0.25))
axes[2].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='response_time_diff', hue='response_switch', data=plot_CsTs_df, ax=axes[3]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='response_time_diff', data=plot_CsTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[3], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CsTs_df[plot_CsTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[3].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['response_time_diff'])[0] - list(subject_data['response_time_diff'])[1] < 0:
        axes[3].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[3].plot(subject_data['response_switch'], subject_data['response_time_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[3].set_title('Context Switch\nTask Switch')
axes[3].set_ylim((-0.25,0.25))
axes[3].get_legend().remove()


# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
custom_palette = ['#555555', '#D55E00']
sns.set_palette(custom_palette)
# group by subject
tmp_df = rt_plot_df[((rt_plot_df['task']!='T3') & (rt_plot_df['prev_task']!='T3'))]

plot_CrCs_df = tmp_df.groupby(['subject','context_switch','task_switch','response_switch'])['zscored_RT_diff'].mean().reset_index() # get avg value of category per subject

plot_Cr_df = plot_CrCs_df[plot_CrCs_df['context_switch']==0]
plot_Cs_df = plot_CrCs_df[plot_CrCs_df['context_switch']==1]

plot_CrTr_df = plot_Cr_df[plot_Cr_df['task_switch']==0]
plot_CrTs_df = plot_Cr_df[plot_Cr_df['task_switch']==1]
plot_CsTr_df = plot_Cs_df[plot_Cs_df['task_switch']==0]
plot_CsTs_df = plot_Cs_df[plot_Cs_df['task_switch']==1]
# set up figure for subplots
fig, axes = plt.subplots(nrows=1, ncols=4, sharey=True, figsize=(16, 7.5)) # set up subplots


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='zscored_RT_diff', hue='response_switch', data=plot_CrTr_df, ax=axes[0]) # boxprops=dict(alpha=.75), , order=[0,1]
sns.stripplot(x='response_switch', y='zscored_RT_diff', data=plot_CrTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[0], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CrTr_df[plot_CrTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[0].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT_diff'])[0] - list(subject_data['zscored_RT_diff'])[1] < 0:
        axes[0].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[0].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[0].set_title('Context Repeat\nTask Repeat')
axes[0].set_ylim((-0.75,0.75))
axes[0].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='zscored_RT_diff', hue='response_switch', data=plot_CrTs_df, ax=axes[1]) # boxprops=dict(alpha=.75),  
sns.stripplot(x='response_switch', y='zscored_RT_diff', data=plot_CrTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[1], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CrTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CrTs_df[plot_CrTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[1].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT_diff'])[0] - list(subject_data['zscored_RT_diff'])[1] < 0:
        axes[1].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[1].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[1].set_title('Context Repeat\nTask Switch')
axes[1].set_ylim((-0.75,0.75))
axes[1].get_legend().remove()


# subplot 1 will be accuracy
#sns.barplot(x='hierarchical_switch_type', y='accuracy', data=plot_acc_df, ax=axes[0], alpha=0.75, order=['Stay','IDS','EDS'])
sns.boxplot(x='response_switch', y='zscored_RT_diff', hue='response_switch', data=plot_CsTr_df, ax=axes[2]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='zscored_RT_diff', data=plot_CsTr_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[2], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTr_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CsTr_df[plot_CsTr_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[2].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT_diff'])[0] - list(subject_data['zscored_RT_diff'])[1] < 0:
        axes[2].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[2].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[2].set_title('Context Switch\nTask Repeat')
axes[2].set_ylim((-0.75,0.75))
axes[2].get_legend().remove()

# subplot 2 will be response initiation
sns.boxplot(x='response_switch', y='zscored_RT_diff', hue='response_switch', data=plot_CsTs_df, ax=axes[3]) # boxprops=dict(alpha=.75), 
sns.stripplot(x='response_switch', y='zscored_RT_diff', data=plot_CsTs_df, color=".1", jitter=0.025, alpha=0.35, ax=axes[3], order=[0,1])
# Manually draw lines between subjects
for subject_id in plot_CsTs_df['subject'].unique():
    if int(subject_id) == 11074:
        continue
    subject_data = plot_CsTs_df[plot_CsTs_df['subject'] == subject_id].sort_values(by='response_switch')
    #axes[3].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='gray', linestyle='--', alpha=0.45, zorder=0)
    if list(subject_data['zscored_RT_diff'])[0] - list(subject_data['zscored_RT_diff'])[1] < 0:
        axes[3].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='peachpuff', linestyle='--', alpha=0.45, zorder=0)
    else:
        axes[3].plot(subject_data['response_switch'], subject_data['zscored_RT_diff'], color='lightblue', linestyle='--', alpha=0.45, zorder=0)
axes[3].set_title('Context Switch\nTask Switch')
axes[3].set_ylim((-0.75,0.75))
axes[3].get_legend().remove()


# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

# PUPIL

In [ ]:
# cell for loading each subject's data frame files (remember we saved out blocks individually)
rel_cue_onset_time = 950 # time in ms 
edfs = []
for sub in DM_df_usable['subject'].unique(): 
    if int(sub) == 11070:
        continue # no eye tracking for this subject
    print("loading subject ", sub)
    df_sub = master_df[master_df['subject']==sub]
    sub_edfs = []  
    for b in range(12):
        file = glob.glob(os.path.join(edf_dir, f"cue__{sub}_{b}.xls"))
        #print(file)
        if file:
            m_blk_edfs=[]
            # load edf block file
            full_edf = pd.read_csv(file[0], encoding='utf-16', sep='\t')
            full_edf["subject"] = sub
            full_edf["block"] = int(b+1)
            full_edf["trial"] = full_edf["TRIAL_INDEX"]
            # now fix pupil diameter 
            is_numeric = pd.to_numeric(full_edf['AVERAGE_PUPIL_SIZE'], errors='coerce').notna()
            pupil_list = full_edf['AVERAGE_PUPIL_SIZE'].copy()
            pupil_list[~is_numeric] = np.nan
            pupil_vec = np.asarray(pupil_list)
            full_edf['AVERAGE_PUPIL_SIZE'] = np.where(full_edf['AVERAGE_PUPIL_SIZE']==0, np.nan, pupil_vec)
            full_edf['AVERAGE_PUPIL_SIZE'] = full_edf['AVERAGE_PUPIL_SIZE'].astype(float)
            # reformat timestamp to be relative to start of epoch
            for trl in full_edf['trial'].unique():
                df_trl = full_edf[full_edf['trial']==trl].copy()
                first_tp = float(list(df_trl['TIMESTAMP'])[0]) # assume it is ordered
                time_vec = np.asarray(df_trl['TIMESTAMP'])
                new_time_vec = time_vec - first_tp - rel_cue_onset_time
                df_trl['timepoint'] = new_time_vec
            
                m_blk_edfs.append(df_trl)
            block_edf = pd.concat(m_blk_edfs, ignore_index=True)
            
            # load data frame will full info (behavioral data frame)
            df_cblk = df_sub[df_sub['block']==int(b+1)]
            
            # merge the two
            merged_df = pd.merge(block_edf, df_cblk, how='left', on=['subject','block','trial'])
            # if any(np.isnan(merged_df['context_run_len'])):
            #     print("ERROR NAN !!!!!!!!")
            #     print(np.where(np.isnan(merged_df['context_run_len'])))
                
            # add merged to list of subejct dfs
            sub_edfs.append(merged_df)
        else:
            print(f"no file for {sub} block {b}")
        
    sub_edf = pd.concat(sub_edfs, ignore_index=True)
    # if any(np.isnan(sub_edf['context_run_len'])):
    #     print("ERROR NAN !!!!!!!!")
    sub_edf['TRIAL_NUM'] = sub_edf['trial'] + (sub_edf['block'].max()*(sub_edf['block']-1))
        
    edfs.append(sub_edf)
    
edf = pd.concat(edfs)
edf = edf.reset_index()
#edf = edf.set_index("sub")
edf

In [ ]:
edf_plot = edf[edf['trial']>2] #.reset_index()
#edf_plot = edf_plot[edf_plot['timepoint']<=3500]
edf_plot

In [ ]:
edf_plot = edf_plot[edf_plot['accuracy']>0] #.reset_index()
edf_plot = edf_plot[edf_plot['prev_accuracy']>0] #.reset_index()
#edf_plot = edf_plot[edf_plot['timepoint']<=3500]
edf_plot

In [ ]:
# interpolate over blinks and saccaddes
sampling_rate = 500 # samples every 2 ms
cue_onset = 950 # cue onsets 400 ms after start of epoch
baseline_win = [-945, -295] # baseline window should be neg. since prior to cue onset, which we will set at 0
interp_win = [-5,5] # 10ms from either side
smoothing_win = 5 # ms
smooth_idx = int(np.floor(smoothing_win/(1000/sampling_rate)/2))


# EPOCH (10ms window)
# reshape timepoint to [epoch x samplept]
timept_vec = np.asarray(edf_plot['timepoint'])
trial_boundaries = np.where(timept_vec == int(0 - cue_onset))[0]
timept_mat = timept_vec.reshape(len(trial_boundaries), int(trial_boundaries[1]-trial_boundaries[0]))
# reshape pupil to [epoch x samplept]
pupil_vec = np.asarray(edf_plot['AVERAGE_PUPIL_SIZE'])
pupil_mat = pupil_vec.reshape(len(trial_boundaries), int(trial_boundaries[1]-trial_boundaries[0])).copy()
# reshape saccade to [epoch x samplept]
saccade_vec = np.asarray(edf_plot['AVERAGE_IN_SACCADE'])
saccade_mat = saccade_vec.reshape(len(trial_boundaries), int(trial_boundaries[1]-trial_boundaries[0]))
# reshape blink to [epoch x samplept]
blink_vec = np.asarray(edf_plot['AVERAGE_IN_BLINK'])
blink_mat = blink_vec.reshape(len(trial_boundaries), int(trial_boundaries[1]-trial_boundaries[0]))


# INTERPOLATE
reject_mat = np.zeros(pupil_mat.shape)
for trl in range(pupil_mat.shape[0]):
    c_trl_vec = pupil_mat[trl,:].copy()
    
    # Saccades
    sacd_vec = np.where(saccade_mat[trl,:]==1)[0] # indices (cols) where there was saccade
    # see if we should just reject
    if saccade_mat[trl,:].sum()/len(saccade_mat[trl,:]) > 0.01 :
        print("saccade proportion: ", round(saccade_mat[trl,:].sum()/len(saccade_mat[trl,:]), 2))
        reject_mat[trl,:] = 1
    # Blinks
    blnk_vec = np.where(blink_mat[trl,:]==1)[0]
    # see if we should just reject
    if blink_mat[trl,:].sum()/len(blink_mat[trl,:]) > 0.1 :
        print("blink proportion: ", round(blink_mat[trl,:].sum()/len(blink_mat[trl,:]), 2))
        reject_mat[trl,:] = 1
    # NaN values
    nan_inds = np.isnan(c_trl_vec)
    nan_vec = np.where(nan_inds)[0]
    # all periods to interpolate
    interp_vec = np.unique( np.concatenate( [sacd_vec, blnk_vec, nan_vec] ) ) # only keep unique indices
    if len(interp_vec)/len(blink_mat[trl,:]) > 0.1 :
        print("interp proportion: ", round(len(interp_vec)/len(blink_mat[trl,:]), 2))
        reject_mat[trl,:] = 1
    
    # actually interpolate if values
    if interp_vec.size > 0:
        interp_deriv = (interp_vec[1:] - interp_vec[:-1]) - 1
        if interp_deriv.sum()>0:
            #print(trl)
            interp_bounds = np.where(interp_deriv>0)[0]
            prev_nb = 0
            nan_chunks = []
            for nb in interp_bounds:
                nan_chunks.append([interp_vec[prev_nb], interp_vec[nb]])
                prev_nb = nb
            nan_chunks.append([interp_vec[prev_nb], int(interp_vec[-1]+1)])
            for nan_chunk in nan_chunks:
                min_idx = max(0, int(nan_chunk[0] + interp_win[0]))
                max_idx = min(int(nan_chunk[-1] + interp_win[1]), len(c_trl_vec))
                replace_val = np.nanmean( c_trl_vec[ min_idx:max_idx ] )
                pupil_mat[trl, int(nan_chunk[0]):int(nan_chunk[-1])] = replace_val # replace nans
        else:
            min_idx = max(0, int(interp_vec[0] + interp_win[0]))
            max_idx = min(int(interp_vec[-1] + interp_win[1]), len(c_trl_vec))
            replace_val = np.nanmean( c_trl_vec[ min_idx:max_idx ] )
            pupil_mat[trl,interp_vec] = replace_val # replace nans
            
# SMOOTH (10ms window)
# padd pupil matrix epochs
padded_pupil_mat = np.concatenate((np.repeat(pupil_mat[:,0:1], smooth_idx, axis=1), pupil_mat), axis=1)
padded_pupil_mat =  np.concatenate((padded_pupil_mat, np.repeat(padded_pupil_mat[:,(padded_pupil_mat.shape[1]-1):padded_pupil_mat.shape[1]], smooth_idx, axis=1)), axis=1)
padded_pupil_mat
# now that we have added padding, smooth
smoothed_pupil_mat = pupil_mat.copy()
for tp in range(smooth_idx,int(padded_pupil_mat.shape[1]-smooth_idx)):
    c_row_means = np.nanmean(padded_pupil_mat[:, int(tp-smooth_idx):int(tp+1+smooth_idx)], axis=1)
    smoothed_pupil_mat[:,int(tp-smooth_idx)] = c_row_means

# # BASELINE CORRECT 
epoch_means = np.nanmean(smoothed_pupil_mat[:,((timept_mat[0,:] > baseline_win[0])&(timept_mat[0,:] < baseline_win[1]))], axis=1)
#epoch_means = np.median(smoothed_pupil_mat[:,((timept_mat[0,:] > 0)&(timept_mat[0,:] < 100))], axis=1)
reject_mat[np.isnan(epoch_means),:] = 1
epoch_means[np.isnan(epoch_means)] = 0 # replace any nan baselines with 0
bcs_pupil_mat = smoothed_pupil_mat - np.repeat(epoch_means[:,np.newaxis], smoothed_pupil_mat.shape[1], axis=1)

edf_plot['baseline_corrected_smoothed_pupil_size'] = bcs_pupil_mat.flatten()
edf_plot['reject_epoch'] = reject_mat.flatten()

In [ ]:
quality_check = edf_plot.groupby(['subject','block','hierarchical_switch_type'])['reject_epoch'].agg(lambda x: sum(x)/1350).reset_index()
quality_check

In [ ]:
pupil_subjs = DM_df_usable['subject'].unique()
print(len(pupil_subjs), " subjects to be included: ", pupil_subjs)

edf_plot['hierarchical_switch_type'] = pd.Categorical(edf_plot['hierarchical_switch_type'], categories=['Stay','IDS','EDS'], ordered=True)
#edf_plot['task_switch'] = pd.Categorical(edf_plot['task_switch'], categories=[0,1], ordered=True) # ["repeat","switch"]
#edf_plot['context_switch'] = pd.Categorical(edf_plot['context_switch'], categories=[0,1], ordered=True) # ["repeat","switch"]

usable_epochs = edf_plot[edf_plot['reject_epoch']==0]
usable_epochs = usable_epochs[((usable_epochs['timepoint'] >= -900)&(usable_epochs['timepoint'] <= 1750))]


In [ ]:
fig, axes = plt.subplots(nrows=len(pupil_subjs), ncols=4, figsize=(20, int(len(pupil_subjs)*3)), sharex=True, sharey=True) # set up subplots

usable_epochs2 = usable_epochs[((usable_epochs['timepoint'] >= -100)&(usable_epochs['timepoint'] <= 1750))]
#usable_epochs2 = usable_epochs2[usable_epochs2['block'] < 7]

# -- loop through subjects
for row_idx,sub in enumerate(DM_df_usable['subject'].unique()):
    plot_df = usable_epochs2[usable_epochs2['subject']==sub]
    
    # if int(sub)==11067 or int(sub)==11085 or int(sub)==11070 or int(sub)==11090:
    #     continue
    
    sns.lineplot(
        data=plot_df, 
        x="timepoint", 
        y="baseline_corrected_smoothed_pupil_size", 
        hue="hierarchical_switch_type", ax=axes[row_idx][0], palette=["#009E73","#56B4E9","#E69F00"], legend=False)
    axes[row_idx][0].set_ylim(-200, 350)
    axes[row_idx][0].set_title(str(sub))
    #axes[row_idx][0].legend_.remove()
    legend = axes[row_idx][0].get_legend()
    if legend is not None:
        legend.remove()
    
    sns.lineplot(
        data=plot_df, 
        x="timepoint", 
        y="baseline_corrected_smoothed_pupil_size", 
        hue="response_switch", ax=axes[row_idx][1], palette=["#555555","#E96056"], legend=False)
    axes[row_idx][1].set_ylim(-200, 350)
    axes[row_idx][1].set_title(str(sub))
    #axes[row_idx][1].legend_.remove()
    legend = axes[row_idx][1].get_legend()
    if legend is not None:
        legend.remove()
        
    sns.lineplot(
        data=plot_df, 
        x="timepoint", 
        y="baseline_corrected_smoothed_pupil_size", 
        hue="task_switch", ax=axes[row_idx][2], palette=["#555555","#E7E956"], legend=False)
    axes[row_idx][2].set_ylim(-200, 350)
    axes[row_idx][2].set_title(str(sub))
    #axes[row_idx][1].legend_.remove()
    legend = axes[row_idx][2].get_legend()
    if legend is not None:
        legend.remove()
    
    # sns.lineplot(
    #     data=plot_df, 
    #     x="timepoint", 
    #     y="baseline_corrected_smoothed_pupil_size", 
    #     hue="context_switch", ax=axes[row_idx][3], palette=["#009E8E","#E69F00"], legend=False)
    # axes[row_idx][3].set_ylim(-200, 350)
    # axes[row_idx][3].set_title(str(sub))
    # #axes[row_idx][2].legend_.remove()
    # legend = axes[row_idx][3].get_legend()
    # if legend is not None:
    #     legend.remove()
        
    # sns.lineplot(
    #     data=plot_df, 
    #     x="timepoint", 
    #     y="baseline_corrected_smoothed_pupil_size", 
    #     hue="prev_accuracy", ax=axes[row_idx][3], palette=["#009E8E","#E96056"], legend=False)
    # axes[row_idx][3].set_ylim(-200, 350)
    # axes[row_idx][3].set_title(str(sub))
    # #axes[row_idx][2].legend_.remove()
    # legend = axes[row_idx][3].get_legend()
    # if legend is not None:
    #     legend.remove()
    
plt.show()
    
# sns.lineplot(
#     data=edf_plot, 
#     x="timepoint", 
#     y="AVERAGE_PUPIL_SIZE", 
#     hue="hierarchical_switch_type",       # Differentiates the 3 conditions
#     errorbar="sd",     # Default: 95% Confidence Interval
#     err_style="band"   # Draws shaded bands instead of bars
# )
# 
# plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=len(pupil_subjs), ncols=12, figsize=(24, int(len(pupil_subjs)*3)), sharex=True, sharey=True) # set up subplots

usable_epochs2 = usable_epochs[((usable_epochs['timepoint'] >= -100)&(usable_epochs['timepoint'] <= 1750))]

# -- loop through subjects
for row_idx,sub in enumerate(DM_df_usable['subject'].unique()):
    for col_idx,blk in enumerate(DM_df_usable['block'].unique()):
        plot_df = usable_epochs2[usable_epochs2['subject']==sub]

        sns.lineplot(
            data=plot_df[plot_df['block']==blk], 
            x="timepoint", 
            y="baseline_corrected_smoothed_pupil_size", 
            hue="hierarchical_switch_type", ax=axes[row_idx][col_idx], palette=["#009E73","#56B4E9","#E69F00"], legend=False)
        axes[row_idx][col_idx].set_ylim(-150, 150)
        axes[row_idx][col_idx].set_title(str(sub))
    
plt.show()

In [ ]:
# put all subjects together into a single plot
# 11052, 11066, 11071, 11079

plot_df = usable_epochs[((usable_epochs['subject']=='11052')|(usable_epochs['subject']=='11066')|(usable_epochs['subject']=='11071')|(usable_epochs['subject']=='11079'))].groupby(['subject','timepoint','hierarchical_switch_type'])['baseline_corrected_smoothed_pupil_size'].mean().reset_index()

sns.lineplot(
            data=plot_df, #usable_epochs, 
            x="timepoint", 
            y="baseline_corrected_smoothed_pupil_size", 
            hue="hierarchical_switch_type", palette=["#009E73","#56B4E9","#E69F00"], 
            legend=False
            )
plt.ylim((-50, 250))
plt.ylabel('change in pupil diameter\n(from baseline [-925 -125])', fontsize=15)
plt.title(f"Hierarchical Switch Costs (only {len(pupil_subjs)} subjects)", fontsize=15)
#plt.gca().get_legend().remove()

plt.show()

In [ ]:
#from scipy import stats
# rt_plot_df.groupby(['subject','context_switch','task_switch','response_switch'])['response_time'].count().reset_index() # get avg value of category per subject
usable_epochs_EDS = usable_epochs[usable_epochs['hierarchical_switch_type'] == "EDS" ].groupby(['subject','timepoint'])['baseline_corrected_smoothed_pupil_size'].mean().reset_index()
usable_epochs_IDS = usable_epochs[usable_epochs['hierarchical_switch_type'] == "IDS" ].groupby(['subject','timepoint'])['baseline_corrected_smoothed_pupil_size'].mean().reset_index()
usable_epochs_Stay = usable_epochs[usable_epochs['hierarchical_switch_type'] == "Stay" ].groupby(['subject','timepoint'])['baseline_corrected_smoothed_pupil_size'].mean().reset_index()

EDS_vs_IDS = np.zeros( (usable_epochs['timepoint'].unique().shape[0], 2) )
EDS_vs_Stay = np.zeros( (usable_epochs['timepoint'].unique().shape[0], 2) )

count = -1
for tp in usable_epochs['timepoint'].unique():
    usable_epochs_EDS_tmp = np.asarray(usable_epochs_EDS[usable_epochs_EDS['timepoint']==tp]['baseline_corrected_smoothed_pupil_size'])
    usable_epochs_IDS_tmp = np.asarray(usable_epochs_IDS[usable_epochs_IDS['timepoint']==tp]['baseline_corrected_smoothed_pupil_size'])
    usable_epochs_Stay_tmp = np.asarray(usable_epochs_Stay[usable_epochs_Stay['timepoint']==tp]['baseline_corrected_smoothed_pupil_size'])
    #print(usable_epochs_EDS_tmp)
    #print(usable_epochs_IDS_tmp)
    #print(len(usable_epochs_Stay_tmp))
    count += 1
    EDS_vs_IDS[count,0], EDS_vs_IDS[count,1] = stats.ttest_ind(a=usable_epochs_EDS_tmp, b=usable_epochs_IDS_tmp, equal_var=False, nan_policy='omit') # t=0, p=1
    EDS_vs_Stay[count,0], EDS_vs_Stay[count] = stats.ttest_ind(a=usable_epochs_EDS_tmp, b=usable_epochs_Stay_tmp, equal_var=False, nan_policy='omit') # t=0, p=1

In [ ]:
EDS_vs_IDS[EDS_vs_IDS[:,1]<0.15]

In [ ]:
# put all subjects together into a single plot
sns.lineplot(
            data=usable_epochs, 
            x="timepoint", 
            y="baseline_corrected_smoothed_pupil_size", 
            hue="response_switch", palette=["#555555","#E96056"], 
            legend=False
            )
plt.ylim((-50, 150))
plt.ylabel('change in pupil diameter\n(from baseline [-900 -300])', fontsize=15)
plt.title(f"Response Switch Costs (only {len(pupil_subjs)} subjects)", fontsize=15)
#plt.gca().get_legend().remove()

plt.show()

In [ ]:
# put all subjects together into a single plot
sns.lineplot(
            data=usable_epochs, 
            x="timepoint", 
            y="baseline_corrected_smoothed_pupil_size", 
            hue="task_switch", palette=["#555555","#E7E956"], 
            legend=False
            )
plt.ylim((-150, 150))
plt.ylabel('change in pupil diameter\n(from baseline [-900 -300])', fontsize=15)
plt.title(f"Task Switch Costs (only {len(pupil_subjs)} subjects)", fontsize=15)
#plt.gca().get_legend().remove()

plt.show()

In [ ]:
# put all subjects together into a single plot
sns.lineplot(
            data=usable_epochs, 
            x="timepoint", 
            y="baseline_corrected_smoothed_pupil_size", 
            hue="context_switch", palette=["#009E73","#E69F00"], 
            legend=False
            )
plt.ylim((-150, 150))
plt.ylabel('change in pupil diameter\n(from baseline [-900 -300])', fontsize=15)
plt.title(f"Context Switch Costs (only {len(pupil_subjs)} subjects)", fontsize=15)
#plt.gca().get_legend().remove()

plt.show()

In [ ]:
fig, axes = plt.subplots(nrows=len(pupil_subjs), ncols=5, figsize=(24, int(len(pupil_subjs)*3)), sharex=True, sharey=True) # set up subplots

#usable_epochs2 = usable_epochs[((usable_epochs['timepoint'] >= -100)&(usable_epochs['timepoint'] <= 1750))]

# -- loop through subjects
for row_idx,sub in enumerate(DM_df_usable['subject'].unique()):
    for col_idx,blk in enumerate(usable_epochs2['context_count'].unique()):
        plot_df = usable_epochs2[usable_epochs2['subject']==sub]

        sns.lineplot(
            data=plot_df[plot_df['block']==blk], 
            x="timepoint", 
            y="baseline_corrected_smoothed_pupil_size", 
            hue="hierarchical_switch_type", ax=axes[row_idx][col_idx], palette=["#009E73","#56B4E9","#E69F00"], legend=False)
        axes[row_idx][col_idx].set_ylim(-150, 150)
        axes[row_idx][col_idx].set_title(str(sub))
    
plt.show()